# Практическая работа. Метод k ближайших соседей (kNN) с нуля
## «Скажи мне, кто твой сосед, – и я скажу, кто ты»

**Цель:** написать метод k ближайших соседей на чистом Python, научить его отличать пешехода, велосипедиста и автомобиль по данным датчиков беспилотника, выяснить, зачем масштабировать признаки и как выбрать число соседей $k$, а в конце применить тот же метод к регрессии – предсказанию тормозного пути.

> **Правило работы.** Только встроенный Python: списки, словари, циклы, `zip`, `sum`, `min`, `max`, `abs`, `sorted`, `len`. Никаких `import` внутри своих функций – ни `numpy`, ни `sklearn`, ни `math`: корень считаем как `** 0.5`. Модуль `random` используется только в готовых ячейках для генерации данных. Библиотеки `matplotlib` и `sklearn` появляются лишь в необязательных готовых ячейках – для графиков и сверки.

**Как работать с ноутбуком.** В ячейках с заданиями замените `...` своим кодом – подсказка написана в комментарии `# <--`. После функций идут строки проверки: напечатанное значение должно совпасть с тем, что стоит после слова «ожидается». Готовые ячейки (без `...`) просто выполните. Сначала прочитайте теорию ниже – все задания опираются на нее.

## Теория: как работает kNN

**1. Идея.** Метод опирается на **гипотезу компактности**: похожие объекты лежат рядом и обычно относятся к одному классу. Поэтому, чтобы узнать класс нового объекта, достаточно посмотреть на его ближайших «соседей» из обучающей выборки.

**2. Алгоритм в четыре шага.** Для каждого нового объекта $x$:

1. посчитать расстояние от $x$ до **всех** объектов обучающей выборки;
2. отсортировать объекты по расстоянию;
3. взять $k$ ближайших (число $k$ выбираем заранее);
4. **классификация:** ответ – класс, который встречается среди соседей чаще всего (голосование, «мода»); **регрессия:** ответ – среднее значение соседей.

«Обучение» kNN – это просто запоминание выборки: модель ничего не подбирает, в отличие от логистической регрессии. Вся работа происходит в момент прогноза. Поэтому kNN называют **ленивым** методом.

**3. Расстояние.** Объект с $n$ признаками – это точка $a = (a_1, \dots, a_n)$. Самые популярные расстояния:

$$\text{евклидово:}\quad d(a, b) = \sqrt{(a_1 - b_1)^2 + (a_2 - b_2)^2 + \dots + (a_n - b_n)^2} = \sqrt{\sum_{i=1}^{n} (a_i - b_i)^2},$$

$$\text{манхэттенское:}\quad d(a, b) = |a_1 - b_1| + |a_2 - b_2| + \dots + |a_n - b_n| = \sum_{i=1}^{n} |a_i - b_i|.$$

Евклидово – длина отрезка «по прямой» (теорема Пифагора), манхэттенское – путь «по клеткам», как такси в квартале. *Пример:* $a = (1, 1)$, $b = (4, 5)$: евклидово $\sqrt{3^2 + 4^2} = 5$, манхэттенское $3 + 4 = 7$.

**4. Голосование.** Пусть 5 ближайших соседей имеют метки $[1, 0, 0, 0, 1]$ (метки идут от ближнего к дальнему). Голосов за класс 0 – три, за класс 1 – два, ответ 0. Если голосов поровну (**ничья**), побеждает класс, у которого сосед **ближе**: пройдем по меткам от ближнего к дальнему и вернем первую метку с наибольшим числом голосов. Чтобы для двух классов ничьих не было вовсе, $k$ берут нечетным.

**5. Масштаб признаков.** Расстояние складывает разности по всем признакам, поэтому признак с большими числами «перекрикивает» остальные. Разница в 30 м по расстоянию до объекта затмит разницу в 3 м по длине, хотя именно длина отличает машину от пешехода. Лечение – **нормализация min-max**: каждый признак $j$ переводим в отрезок $[0, 1]$:

$$x'_j = \frac{x_j - \min_j}{\max_j - \min_j}.$$

Минимум и максимум считаем **только по обучающей выборке**, а затем той же формулой переводим и тестовые объекты – как будто тест мы еще не видели. Поэтому у тестовых объектов значения могут чуть выходить за $[0, 1]$ – это нормально.

**6. Как измерить качество.** У kNN нет обучения, а значит, нет и функции потерь, которую метод минимизирует. Но ошибку прогнозов мы измерять обязаны:

$$\text{доля верных ответов:}\quad \mathrm{accuracy} = \frac{\text{число верных ответов}}{N}, \qquad \text{доля ошибок} = 1 - \mathrm{accuracy};$$

$$\text{для регрессии – среднеквадратичная ошибка:}\quad \mathrm{MSE} = \frac{1}{N}\sum_{i=1}^{N} (y_i - \hat y_i)^2, \qquad \mathrm{RMSE} = \sqrt{\mathrm{MSE}}.$$

Здесь $y_i$ – правильный ответ, $\hat y_i$ – прогноз, $N$ – число объектов. RMSE измеряется в тех же единицах, что и ответ (например, в метрах), поэтому его удобно объяснять словами.

**7. Выбор $k$.** При $k = 1$ модель верит одному соседу и повторяет любой шум – это **переобучение**, граница классов получается рваной. При очень большом $k$ голосует почти вся выборка, и модель всегда отвечает самым частым классом – **недообучение**. Лучшее $k$ ищут перебором по **валидационной** выборке – отдельному куску обучающих данных. Тестовую выборку для выбора $k$ трогать нельзя: иначе оценка качества окажется слишком оптимистичной.

In [ ]:
# Готовая ячейка: только встроенные модули Python
import random

RANDOM_STATE = 42

def train_test_split(X, y, test_size=0.3, seed=RANDOM_STATE):
    """Перемешивает объекты и делит их на обучающую и тестовую части."""
    idx = list(range(len(X)))
    random.Random(seed).shuffle(idx)
    n_test = int(len(X) * test_size)
    test, train = idx[:n_test], idx[n_test:]
    return ([X[i] for i in train], [X[i] for i in test],
            [y[i] for i in train], [y[i] for i in test])

def make_road_objects(n_per_class=100, seed=RANDOM_STATE):
    """Объекты, которые «видит» беспилотник: 0 – пешеход, 1 – велосипедист, 2 – автомобиль.
    Признаки: [скорость, км/ч;  длина, м;  высота, м;  расстояние до объекта, м]."""
    rnd = random.Random(seed)
    params = {0: ((5, 2.5), (0.7, 0.4), (1.7, 0.15)),     # (среднее, разброс) скорости, длины, высоты
              1: ((14, 6), (1.7, 0.45), (1.75, 0.15)),
              2: ((18, 12), (3.6, 1.1), (1.6, 0.2))}
    X, y = [], []
    for c, (pv, pl, ph) in params.items():
        for _ in range(n_per_class):
            v = max(0.5, rnd.gauss(*pv))
            L = max(0.2, rnd.gauss(*pl))
            H = max(0.8, rnd.gauss(*ph))
            D = rnd.uniform(5, 80)            # расстояние не зависит от класса объекта
            X.append([round(v, 1), round(L, 2), round(H, 2), round(D, 1)])
            y.append(c)
    idx = list(range(len(X)))
    rnd.shuffle(idx)
    return [X[i] for i in idx], [y[i] for i in idx]

def make_braking(n=80, seed=RANDOM_STATE):
    """Скорость автомобиля (км/ч) -> тормозной путь (м) с учетом реакции водителя 1 с."""
    rnd = random.Random(seed)
    X, y = [], []
    for _ in range(n):
        v = rnd.uniform(10, 120)
        s = v / 3.6 * 1.0 + v ** 2 / (254 * 0.7)
        X.append([round(v, 1)])
        y.append(round(s + rnd.gauss(0, 3 + 0.05 * v), 1))
    return X, y

CLASS_NAMES = ['пешеход', 'велосипедист', 'автомобиль']
X_road, y_road = make_road_objects()
X_train, X_test, y_train, y_test = train_test_split(X_road, y_road)
print(len(X_train), len(X_test), '  ожидается 210 90')
print(X_train[0], CLASS_NAMES[y_train[0]])

In [ ]:
# Учебный пример со слайдов: робот-сортировщик, яблоко (0) или груша (1)?
# Признаки: [ширина, см;  высота, см]
X_fruit = [[7, 6], [8, 7], [7, 5], [9, 8],      # яблоки
           [6, 8], [4, 9], [3, 9], [5, 11]]     # груши
y_fruit = [0, 0, 0, 0, 1, 1, 1, 1]
q = [6, 7]                                      # новый фрукт: что это?

**Задание 1. Расстояния.**

Напишите две функции, которые получают два списка одинаковой длины (два объекта) и возвращают число:

- `euclidean(a, b)` – евклидово расстояние (п. 3 теории): накопите в `s` сумму квадратов разностей координат, затем извлеките корень `s ** 0.5`;
- `manhattan(a, b)` – манхэттенское расстояние: сумма модулей разностей, модуль – встроенная функция `abs`.

Пары координат удобно перебирать через `zip(a, b)`: для `a = [1, 1]`, `b = [4, 5]` цикл `for ai, bi in zip(a, b)` даст пары `(1, 4)` и `(1, 5)`.

*Посчитайте на бумаге до запуска:* расстояние от нового фрукта $q = (6, 7)$ до яблока $(7, 6)$ – это $\sqrt{(6-7)^2 + (7-6)^2} = \sqrt{2} \approx 1{,}414$.

In [ ]:
def euclidean(a, b):
    s = 0
    for ai, bi in zip(a, b):
        s += ...                   # <-- прибавьте квадрат разности координат
    return ...                     # <-- корень из суммы

def manhattan(a, b):
    s = 0
    for ai, bi in zip(a, b):
        s += ...                   # <-- прибавьте модуль разности
    return s


print(euclidean([1, 1], [4, 5]), '  ожидается 5.0')
print(manhattan([1, 1], [4, 5]), '  ожидается 7')
print(euclidean([0, 0, 0, 0], [1, 1, 1, 1]), '  ожидается 2.0 (работает для любого числа признаков)')
print([round(euclidean(p, q), 3) for p in X_fruit])
print('  ожидается [1.414, 2.0, 2.236, 3.162, 1.0, 2.828, 3.606, 4.123]')

**Вопрос 1.** а) Какой фрукт ближе всего к $q$? б) Может ли манхэттенское расстояние быть меньше евклидова для тех же двух точек? Почему? в) Что будет с евклидовым расстоянием, если оба признака записать в миллиметрах вместо сантиметров? Изменится ли порядок соседей?

*Ваш ответ:*

**Задание 2. Соседи и голосование.**

- `get_neighbors(X_train, y_train, x, k, dist=euclidean)` возвращает **метки** $k$ ближайших к `x` объектов, от ближнего к дальнему. Шаг 1: соберите список пар `(расстояние, метка)` – по одной паре на обучающий объект. Шаг 2 дан готовым: `sorted(pairs, key=lambda p: p[0])` сортирует пары по первому элементу – расстоянию. Шаг 3: возьмите первые `k` пар срезом `pairs[:k]` и оставьте от них только метки. Параметр `dist` позволяет подставить любое расстояние – по умолчанию евклидово.
- `majority_vote(labels)` возвращает самую частую метку (п. 4 теории). Голоса считаем в словаре `counts`: ключ – метка, значение – число голосов. Выражение `counts.get(lab, 0)` вернет текущее число голосов или 0, если метка встретилась впервые. При ничьей побеждает метка ближайшего соседа – она раньше встречается в `labels`.

*Разбор на фруктах.* Отсортируем расстояния из задания 1: $1{,}0$ (груша), $1{,}414$ (яблоко), $2{,}0$ (яблоко), $2{,}236$ (яблоко), $2{,}828$ (груша), … При $k = 1$ ответ – груша, при $k = 3$ – яблоко (2 : 1), при $k = 5$ – яблоко (3 : 2).

In [ ]:
def get_neighbors(X_train, y_train, x, k, dist=euclidean):
    pairs = []
    for xi, yi in zip(X_train, y_train):
        pairs.append(...)                   # <-- пара (расстояние от xi до x, метка yi)
    pairs = sorted(pairs, key=lambda p: p[0])  # сортировка по расстоянию
    return ...                              # <-- метки первых k пар (срез pairs[:k], от пары берем только yi)

def majority_vote(labels):
    counts = {}
    for lab in labels:
        counts[lab] = ...                   # <-- добавьте один голос метке lab
    best = ...                              # <-- наибольшее число голосов
    for lab in labels:                         # от ближнего соседа к дальнему
        if counts[lab] == best:
            return lab


print(get_neighbors(X_fruit, y_fruit, q, 5), '  ожидается [1, 0, 0, 0, 1]')
print(majority_vote([1, 0, 0]), '  ожидается 0')
print(majority_vote([2, 1, 1, 2]), '  ожидается 2 (ничья, но сосед класса 2 ближе)')
for k in [1, 3, 5]:
    print('k =', k, '->', majority_vote(get_neighbors(X_fruit, y_fruit, q, k)))
print('  ожидается k = 1 -> 1, k = 3 -> 0, k = 5 -> 0')

**Вопрос 2.** а) Почему ответ при $k = 1$ и $k = 3$ разный? Какому ответу вы бы поверили больше и почему? б) Зачем брать нечетное $k$, если классов два? Спасет ли нечетное $k$ от ничьей, когда классов три?

*Ваш ответ:*

**Задание 3. Прогноз и точность.**

- `knn_predict(X_train, y_train, X_test, k, dist=euclidean)` – прогноз для **каждого** объекта из `X_test`: найдите метки соседей (`get_neighbors`) и проголосуйте (`majority_vote`). Возвращает список меток той же длины, что `X_test`.
- `accuracy(y_true, y_pred)` – доля верных ответов (п. 6 теории).

Затем применим kNN к данным беспилотника. Каждый объект – это 4 числа: скорость (км/ч), длина (м), высота (м) и расстояние до объекта (м). Классы: 0 – пешеход, 1 – велосипедист, 2 – автомобиль. Обучающая выборка – 210 объектов, тестовая – 90.

In [ ]:
def knn_predict(X_train, y_train, X_test, k, dist=euclidean):
    y_pred = []
    for x in X_test:
        labels = ...                        # <-- метки k ближайших соседей
        y_pred.append(...)                  # <-- голосование
    return y_pred

def accuracy(y_true, y_pred):
    correct = 0
    for a, b in zip(y_true, y_pred):
        if a == b:
            correct += 1
    return ...                            # <-- доля верных ответов


print(knn_predict(X_fruit, y_fruit, [q, [8, 6], [4, 10]], k=3), '  ожидается [0, 0, 1]')
print(accuracy([0, 1, 2, 1], [0, 1, 1, 1]), '  ожидается 0.75')

y_pred = knn_predict(X_train, y_train, X_test, k=5)
acc_raw = accuracy(y_test, y_pred)
print(round(acc_raw, 3), '  ожидается 0.744')

In [ ]:
# Готовая ячейка (необязательная): сверка с sklearn, если библиотека установлена
try:
    from sklearn.neighbors import KNeighborsClassifier
    sk = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
    same = accuracy(list(sk.predict(X_test)), y_pred)
    print('совпадение с sklearn:', round(same, 3), '  ожидается 0.956')
    print('расхождения – это ничьи: sklearn при равенстве голосов берет класс с меньшим номером, а мы – класс ближайшего соседа')
except ImportError:
    print('sklearn не установлен – сверку можно пропустить')

**Вопрос 3.** а) Почему kNN почти ничего не делает на этапе «обучения», но долго работает на этапе прогноза? Сколько расстояний мы посчитали для 90 тестовых объектов? б) Точность 0,744 – это хорошо или плохо? С чем ее сравнить? (Подсказка: какой была бы точность, если всегда отвечать «пешеход»?)

*Ваш ответ:*

**Задание 4. Масштаб признаков.**

Посмотрите на признаки: скорость – десятки км/ч, расстояние до объекта – от 5 до 80 м, а длина и высота – единицы метров. В евклидовом расстоянии побеждают большие числа, то есть скорость и расстояние, причем расстояние до объекта **вообще не связано** с классом – пешеход может быть и в 5, и в 80 метрах.

- `minmax_fit(X)` – для каждого признака (столбца) найти минимум и максимум. Столбец `j` дан готовым: `[row[j] for row in X]`. Возвращает два списка: `mins` и `maxs`.
- `minmax_transform(X, mins, maxs)` – перевести каждый объект по формуле $x'_j = (x_j - \min_j) / (\max_j - \min_j)$ из п. 5 теории. Для одной строки удобно так: `[(v - lo) / (hi - lo) for v, lo, hi in zip(row, mins, maxs)]`.

*Пример.* Признак «длина» на обучении меняется от 0,2 до 6,1 м. Машина длиной 4,5 м превратится в $(4{,}5 - 0{,}2) / (6{,}1 - 0{,}2) \approx 0{,}73$.

Эксперимент в три шага: сырые признаки → нормализованные → нормализованные без бесполезного признака «расстояние».

In [ ]:
def minmax_fit(X):
    mins, maxs = [], []
    for j in range(len(X[0])):
        column = [row[j] for row in X]     # все значения признака j
        mins.append(...)                   # <-- минимум столбца
        maxs.append(...)                   # <-- максимум столбца
    return mins, maxs

def minmax_transform(X, mins, maxs):
    X_new = []
    for row in X:
        new_row = ...                       # <-- min-max для каждого признака строки
        X_new.append(new_row)
    return X_new


print(minmax_fit([[1, 10], [3, 30], [2, 20]]), '  ожидается ([1, 10], [3, 30])')
print(minmax_transform([[1, 10], [3, 30], [2, 20]], [1, 10], [3, 30]),
      '  ожидается [[0.0, 0.0], [1.0, 1.0], [0.5, 0.5]]')

# 1) нормализуем все 4 признака: min и max – только по обучающей выборке!
mins, maxs = minmax_fit(X_train)
X_train_s = minmax_transform(X_train, mins, maxs)
X_test_s = minmax_transform(X_test, mins, maxs)
acc_scaled = accuracy(y_test, knn_predict(X_train_s, y_train, X_test_s, k=5))

# 2) убираем признак «расстояние до объекта» (последний столбец)
X_train_3 = [row[:3] for row in X_train]
X_test_3 = [row[:3] for row in X_test]
mins3, maxs3 = minmax_fit(X_train_3)
X_train_3s = minmax_transform(X_train_3, mins3, maxs3)
X_test_3s = minmax_transform(X_test_3, mins3, maxs3)
acc_3 = accuracy(y_test, knn_predict(X_train_3s, y_train, X_test_3s, k=5))

print('сырые признаки:           ', round(acc_raw, 3), '  ожидается 0.744')
print('min-max, 4 признака:      ', round(acc_scaled, 3), '  ожидается 0.844')
print('min-max, без расстояния:  ', round(acc_3, 3), '  ожидается 0.922')

**Вопрос 4.** а) Почему нормализация так сильно подняла точность? б) Почему после удаления признака «расстояние» точность выросла еще, хотя информации стало меньше? в) Почему `minmax_fit` вызывается только на обучающей выборке, а не на всех данных сразу?

*Ваш ответ:*

**Задание 5. Подбор $k$.**

Выбирать $k$ по тестовой выборке нельзя (п. 7 теории), поэтому отрежем от обучающей выборки **валидационную** часть: 147 объектов на обучение, 63 на валидацию.

`choose_k(X_tr, y_tr, X_val, y_val, ks)` – для каждого $k$ из списка `ks` сделайте прогноз на валидации, посчитайте **долю ошибок** $1 - \mathrm{accuracy}$ и сложите ее в список `errors`. Лучшее $k$ – то, у которого ошибка минимальна: `ks[errors.index(min(errors))]` (метод `index` вернет первое, то есть самое маленькое подходящее $k$). Функция возвращает пару `(best_k, errors)`.

После выбора $k$ обучим модель на **всей** обучающей выборке и один раз проверим на тесте.

In [ ]:
def choose_k(X_tr, y_tr, X_val, y_val, ks):
    errors = []
    for k in ks:
        y_val_pred = ...                    # <-- прогноз на валидации
        errors.append(...)                  # <-- доля ошибок
    best_k = ...                            # <-- k с минимальной ошибкой
    return best_k, errors


X_tr, X_val, y_tr, y_val = train_test_split(X_train_3s, y_train, test_size=0.3, seed=13)
ks = list(range(1, 100, 2))
best_k, errors = choose_k(X_tr, y_tr, X_val, y_val, ks)
print(len(X_tr), len(X_val), '  ожидается 147 63')
print('ошибка при k = 1:', round(errors[0], 3), '  ожидается 0.19')
print('лучшее k:', best_k, ' ошибка', round(min(errors), 3), '  ожидается 9 и 0.079')
print('ошибка при k = 99:', round(errors[-1], 3), '  ожидается 0.302')

acc_best = accuracy(y_test, knn_predict(X_train_3s, y_train, X_test_3s, best_k))
print('точность на тесте при лучшем k:', round(acc_best, 3), '  ожидается 0.933')

In [ ]:
# Готовая ячейка (необязательная): график ошибки на валидации
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7, 3.5))
    plt.plot(ks, errors, marker='o', color='#8F8260')
    plt.axvline(best_k, color='#1C1C22', linestyle='--', label=f'лучшее k = {best_k}')
    plt.xlabel('k – число соседей'); plt.ylabel('доля ошибок на валидации')
    plt.title('Слишком мало соседей – шум, слишком много – «средняя температура»')
    plt.legend(); plt.grid(alpha=0.3); plt.show()
except ImportError:
    for k, e in zip(ks, errors):
        print(k, round(e, 3), '#' * int(e * 100))

**Вопрос 5.** а) Опишите форму графика: что происходит слева (маленькие $k$) и справа (большие $k$)? Где переобучение, а где недообучение? б) Что вернет модель при $k = 210$ – когда голосует вся обучающая выборка? в) Почему лучшее $k$ нельзя выбирать по тестовой выборке?

*Ваш ответ:*

**Задание 6. kNN для регрессии: тормозной путь.**

Беспилотнику нужно оценить тормозной путь по скорости. Настоящая формула нам не нужна – есть 80 замеров «скорость → тормозной путь», и kNN предскажет путь как **среднее** путей $k$ машин с самой близкой скоростью (п. 2 теории).

- `knn_regress(X_train, y_train, X_test, k)` – для каждого объекта найдите значения $k$ ближайших соседей и верните их среднее. Функция `get_neighbors` уже подходит: она возвращает `y` соседей, и ей все равно, метки это или числа.
- `mse(y_true, y_pred)` – среднеквадратичная ошибка (п. 6 теории).

*Считаем вместе.* Если три ближайшие по скорости машины остановились через 40, 44 и 45 м, прогноз – $(40 + 44 + 45) / 3 = 43$ м.

In [ ]:
def knn_regress(X_train, y_train, X_test, k):
    y_pred = []
    for x in X_test:
        values = ...                        # <-- ответы k ближайших соседей
        y_pred.append(...)                  # <-- их среднее
    return y_pred

def mse(y_true, y_pred):
    s = 0
    for a, b in zip(y_true, y_pred):
        s += ...                           # <-- квадрат ошибки
    return ...                             # <-- среднее


print(knn_regress([[10], [20], [30], [40]], [5, 9, 16, 24], [[24], [100]], k=2), '  ожидается [12.5, 20.0]')
print(mse([1, 2, 3], [1, 4, 0]), '  ожидается 4.333...')

X_b, y_b = make_braking()
Xb_train, Xb_test, yb_train, yb_test = train_test_split(X_b, y_b)
for k in [1, 5, 15, 50]:
    m = mse(yb_test, knn_regress(Xb_train, yb_train, Xb_test, k))
    print(f'k = {k:2d}   MSE = {m:7.1f}   RMSE = {m ** 0.5:5.1f} м')
print('  ожидается RMSE: 9.3, 6.5, 8.5, 29.3 м')

In [ ]:
# Готовая ячейка (необязательная): как выглядит прогноз kNN-регрессии
try:
    import matplotlib.pyplot as plt
    grid = [[v] for v in range(5, 126)]
    plt.figure(figsize=(7, 3.8))
    plt.scatter([x[0] for x in Xb_train], yb_train, s=18, color='#1C1C22', label='обучающие замеры')
    for k, col in [(1, '#9A9AA3'), (5, '#8F8260'), (50, '#B8AB84')]:
        plt.plot([g[0] for g in grid], knn_regress(Xb_train, yb_train, grid, k), color=col, label=f'k = {k}')
    plt.xlabel('скорость, км/ч'); plt.ylabel('тормозной путь, м')
    plt.legend(); plt.grid(alpha=0.3); plt.show()
except ImportError:
    print('matplotlib не установлен – график можно пропустить')

**Вопрос 6.** а) Почему прогноз kNN-регрессии на графике похож на лесенку? б) Что происходит с прогнозом при $k = 50$ на больших скоростях и почему? в) Что предскажет модель для скорости 200 км/ч, которой нет в данных? Опасно ли это для беспилотника?

*Ваш ответ:*

### Задание повышенной сложности

1. **Взвешенное голосование.** Пусть близкий сосед весит больше дальнего: вес соседа $w = 1 / d$. Напишите `weighted_vote(pairs)`, где `pairs` – список `(расстояние, метка)`, возвращающую класс с наибольшей **суммой весов**. Проверьте на фруктах при $k = 3$ и $k = 5$ (ответ: оба раза яблоко; при $k = 3$ суммы весов: груша 1,0, яблоко $0{,}707 + 0{,}5 = 1{,}207$). Что делать, если $d = 0$?
2. **Косинусное расстояние.** $d(a, b) = 1 - \dfrac{\sum a_i b_i}{\sqrt{\sum a_i^2}\,\sqrt{\sum b_i^2}}$. Чему оно равно для $a = (1, 2)$ и $b = (2, 4)$? Почему? Попробуйте его на нормализованных данных беспилотника.
3. **Проклятие размерности.** Сгенерируйте 200 случайных точек в кубе $[0, 1]^D$ (`random.random()`), возьмите одну точку и найдите отношение расстояния до ближайшей точки к расстоянию до самой дальней для $D = 2, 10, 100, 1000$. Что происходит с этим отношением? Почему при большом числе признаков «ближайший сосед» перестает быть близким?
4. **Сколько стоит прогноз.** Посчитайте, сколько раз вызывается `euclidean` при прогнозе на всем тесте. Прочитайте, как устроены KD-дерево и BallTree (ссылки у преподавателя), и объясните одной фразой, почему они ускоряют поиск соседей.

In [ ]:
# Место для задания повышенной сложности